# MSPC 异常样本交互式清洗（T² / SPE）

人工在环的清洗流程，像 PLS_Toolbox 一样：

1. 拟合 MSPC，画出 T²（Hotelling）vs SPE（Q 残差）散点，正常点蓝色、异常点红色；
2. 人工根据标注的下标决定删除哪些样本；
3. 点「剔除并重算」，重新拟合、重新可视化，直到满意。

依赖：`aimeta`、`matplotlib`；交互控件可选 `ipywidgets`（没装就用文末的手动单元格）。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from aimeta.monitoring import MSPC

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False

In [ ]:
# 载入你的光谱（(n_samples, n_wavelengths)）；这里用合成数据演示
rng = np.random.default_rng(0)
n, p = 120, 60
W = rng.normal(size=(p, 3))
S = rng.normal(size=(n, 3))
X = S @ W.T + rng.normal(0, 0.01, size=(n, p))
X[5] += 8.0   # 注入两个异常作演示
X[17] -= 6.0
X_ORIG = X.copy()

# 换成真实数据，例如：
# X = np.load("Data/Example1/spectra.npy")
# wl = np.load("Data/Example1/wavelengths.npy")   # 可选，画贡献图用
wl = None

In [ ]:
NC = 5
ALPHA = 0.05

def redraw(X):
    m = MSPC(n_components=NC, alpha=ALPHA).fit(X)
    r = m.monitor(X)
    bad = r["t2_alarm"] | r["spe_alarm"]
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.axvline(r["t2_limit"], color="gray", ls="--", lw=1)
    ax.axhline(r["spe_limit"], color="gray", ls="--", lw=1)
    ax.scatter(r["T2"][~bad], r["SPE"][~bad], c="#4a7fb5", s=30,
               label=f"正常 {int((~bad).sum())}")
    ax.scatter(r["T2"][bad], r["SPE"][bad], c="#b3352c", s=70,
               label=f"异常 {int(bad.sum())}")
    order = np.argsort(r["SPE"])[::-1][:10]
    for i in order:
        ax.annotate(str(i), (r["T2"][i], r["SPE"][i]), fontsize=8,
                    xytext=(4, 4), textcoords="offset points")
    ax.set_xlabel("T² (Hotelling)")
    ax.set_ylabel("SPE (Q 残差)")
    ax.set_title(f"T² vs SPE（保留 {len(X)} 样本）")
    ax.legend()
    plt.show()
    return r

def plot_contribution(r, i):
    c = r["contribution"][i]
    x = wl if wl is not None else np.arange(len(c))
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(x, c, color="#b3352c")
    ax.set_xlabel("波长 / nm" if wl is not None else "波长序号")
    ax.set_ylabel("贡献")
    ax.set_title(f"样本 {i} 的 SPE 贡献图")
    plt.show()

In [ ]:
if HAS_WIDGETS:
    drop_text = widgets.Text(placeholder="如 5,17", description="删除下标:")
    btn = widgets.Button(description="剔除并重算", button_style="danger")
    reset = widgets.Button(description="重置")
    out = widgets.Output()
    status = widgets.HTML()

    def on_btn(b):
        global X
        txt = drop_text.value.strip().replace("，", ",")
        idxs = []
        if txt:
            try:
                idxs = [int(s) for s in txt.split(",") if s.strip() != ""]
            except ValueError:
                with out:
                    clear_output(wait=True)
                    print("下标格式不对：用逗号分隔，如 5,17")
                return
        if idxs:
            X = np.delete(X, idxs, axis=0)
        with out:
            clear_output(wait=True)
            redraw(X)
            status.value = f"已删 {len(idxs)} 个，剩余 {len(X)} 样本"
        drop_text.value = ""

    def on_reset(b):
        global X
        X = X_ORIG.copy()
        with out:
            clear_output(wait=True)
            redraw(X)
            status.value = "已重置"

    btn.on_click(on_btn)
    reset.on_click(on_reset)
    display(widgets.HBox([drop_text, btn, reset]), status, out)
    with out:
        redraw(X)
else:
    print("未安装 ipywidgets：请用下方「手动」单元格。")

### 手动方式（无 ipywidgets 时）

改 `drop` 列表后重跑本单元格，即完成一次「删 → 重算 → 重画」。

In [ ]:
drop = []                    # 填要删的下标，如 [5, 17]
if drop:
    X = np.delete(X, drop, axis=0)
r = redraw(X)
plot_contribution(r, int(np.argmax(r["SPE"])))